# Dixon & Robinson モデル（倍率をホーム・アウェイで共通にする）

`dixon_robinson_model.ipynb` のモデルVIを少し変えたもの。

元のモデルでは、スコアによる倍率をホーム用の $\lambda_{xy}$ とアウェイ用の $\mu_{xy}$ に分けていた。
ここでは分けずに、**そのチームから見た状況**（リードしているか、負けているか）に倍率を1つずつ割り当てる。

たとえばスコアが1-0のとき
- 元のモデル：ホームは $\lambda_{10}$、アウェイは $\mu_{10}$
- このモデル：ホームは「1-0でリード」の倍率、アウェイは「0-1で負けている」の倍率

ホームとアウェイの違いは γ_h（ホームアドバンテージ）だけで表すことになる。

## データ

元のノートブックと同じ `*_goals_and_red_cards.csv` を使う。

- 退場の行は使わない
- 0-0の試合は `dr_time` が空の行が1行だけあるので、得点なしの試合として入れる

In [14]:
import numpy as np
import pandas as pd
from scipy.optimize import minimize

## モデル

$$\lambda(t) = \rho(t)\big(\alpha_i\beta_j\gamma_h \cdot \theta_{s_H} + \xi_1 t\big)$$
$$\mu(t) = \rho(t)\big(\alpha_j\beta_i \cdot \theta_{s_A} + \xi_2 t\big)$$

- $s_H$ はホームから見た状況、$s_A$ はアウェイから見た状況
- $\theta$ はホームとアウェイで同じものを使う
- 同点のときは $\theta = 1$
- それ以外（α, β, γ_h, ρ, ξ、尤度の形）は元のモデルと同じ

### 状況の区分（チームから見て）
| 区分 | 意味 | ホームなら | アウェイなら |
|---|---|---|---|
| level（基準） | 同点 | 0-0, 1-1, … | 0-0, 1-1, … |
| lead1 | 1-0でリード | 1-0 | 0-1 |
| behind1 | 0-1で負けている | 0-1 | 1-0 |
| leadBig | 1-0以外でリード | 2-0, 2-1, … | 0-2, 1-2, … |
| behindBig | 0-1以外で負けている | 0-2, 1-2, … | 2-0, 2-1, … |

元のモデルの区分（home1, away1, homeBig, awayBig）を、チーム目線で言い直しただけ。
倍率は元の8個（λ が4個、μ が4個）から4個に減る。

In [15]:
# 前半・後半の最後の1分（ρ1, ρ2 を掛ける区間）
INJ1_START, INJ1_END = 44 / 90, 45 / 90
INJ2_START, INJ2_END = 89 / 90, 90 / 90

# 基準（level）以外の状況。パラメータはこの順に並べる
STATES = ["lead1", "behind1", "leadBig", "behindBig"]
N_STATES = len(STATES)


class _InvalidRate(Exception):
    """得点強度が0以下になったときに出すエラー"""
    pass


def _team_states(x, y):
    """スコア (x, y) から、(ホームから見た状況, アウェイから見た状況) を返す"""
    if x == y:
        return "level", "level"
    if (x, y) == (1, 0):
        return "lead1", "behind1"
    if (x, y) == (0, 1):
        return "behind1", "lead1"
    if x > y:
        return "leadBig", "behindBig"
    return "behindBig", "leadBig"

## データの前処理

元のノートブックと同じ。csvを試合ごとの得点の並びにする。

In [16]:
def prepare_matches(df):
    """csvを試合ごとの得点の並びに変える。
    戻り値: [{"home", "away", "events": [(時刻, "H_GOAL" か "A_GOAL"), ...]}, ...]
    """
    matches = {}
    d = df.sort_values(["match_id", "dr_time"])
    for row in d.itertuples(index=False):
        mid = row.match_id
        if mid not in matches:
            matches[mid] = {"home": row.home_team, "away": row.away_team, "events": []}
        if pd.isna(row.dr_time):
            # 0-0の試合の行（得点なし）
            continue
        if pd.notna(row.red_card):
            # 退場の行は使わない
            continue
        t = float(row.dr_time)
        side = int(row.home_away)  # 得点したチーム（0=home, 1=away）
        kind = "H_GOAL" if side == 0 else "A_GOAL"
        matches[mid]["events"].append((t, kind))
    return list(matches.values())


def build_index(matches):
    teams = sorted(set([m["home"] for m in matches] + [m["away"] for m in matches]))
    idx = {t: i for i, t in enumerate(teams)}
    return teams, idx

## 尤度関数

`theta` からパラメータを取り出して、対数尤度を計算する。

- チーム以外のパラメータは `gamma_h, rho1, rho2`、倍率4個、`xi1, xi2` の9個
- α, β, γ_h, ρ, 倍率は正でないといけないので `exp()` をかける
- 得点強度が0以下になったらペナルティを付ける

In [17]:
PARAM_NAMES_GLOBAL = ["gamma_h", "rho1", "rho2"] + [f"theta_{s}" for s in STATES] + ["xi1", "xi2"]
N_GLOBAL = len(PARAM_NAMES_GLOBAL)


def unpack_params(theta, n_teams):
    a = theta[0:n_teams]
    b = theta[n_teams:2 * n_teams]
    g = theta[2 * n_teams:]

    alpha = np.exp(a)
    alpha = alpha / alpha.mean()      # 攻撃力の平均を1にする
    beta = np.exp(b)

    gamma_h = np.exp(g[0])
    rho1 = np.exp(g[1])
    rho2 = np.exp(g[2])
    th = {"level": 1.0}               # 同点は1に固定
    for k, s in enumerate(STATES):
        th[s] = np.exp(g[3 + k])
    xi1, xi2 = g[-2], g[-1]
    return alpha, beta, gamma_h, rho1, rho2, th, xi1, xi2

In [18]:
def _check_pos(base, xi, t):
    if base + xi * t <= 0:
        raise _InvalidRate()


def _integrate_rate(t1, t2, base, xi, rho1, rho2):
    """区間 [t1, t2] で rho(t)*(base + xi*t) を積分する（rho が変わる時刻で区間を分ける）"""
    bpoints = sorted({t1, t2} | {p for p in (INJ1_START, INJ1_END, INJ2_START, INJ2_END) if t1 < p < t2})
    for p in bpoints:
        _check_pos(base, xi, p)
    total = 0.0
    for a, c in zip(bpoints[:-1], bpoints[1:]):
        mid = 0.5 * (a + c)
        if INJ1_START < mid <= INJ1_END:
            r = rho1
        elif INJ2_START < mid <= INJ2_END:
            r = rho2
        else:
            r = 1.0
        total += r * (base * (c - a) + xi * (c ** 2 - a ** 2) / 2.0)
    return total


def _rate_at(t, base, xi, rho1, rho2):
    """時刻 t の強度 rho(t)*(base + xi*t)"""
    val = base + xi * t
    if val <= 0:
        raise _InvalidRate()
    if INJ1_START < t <= INJ1_END:
        r = rho1
    elif INJ2_START < t <= INJ2_END:
        r = rho2
    else:
        r = 1.0
    return r * val


def _match_loglik(match, idx, alpha, beta, gamma_h, rho1, rho2, th, xi1, xi2):
    hi, aj = idx[match["home"]], idx[match["away"]]
    lam_k = alpha[hi] * beta[aj] * gamma_h   # ホームの基礎の強度（同点のとき）
    mu_k = alpha[aj] * beta[hi]              # アウェイの基礎の強度（同点のとき）

    x = y = 0     # 今のスコア
    t_prev = 0.0
    ll = 0.0

    for t_ev, kind in sorted(match["events"], key=lambda e: e[0]):
        sh, sa = _team_states(x, y)

        # ホームもアウェイも同じ表 th から倍率を取り出す
        base_h = lam_k * th[sh]
        base_a = mu_k * th[sa]

        # 前の得点から今の得点までの区間の積分を引く
        ll -= _integrate_rate(t_prev, t_ev, base_h, xi1, rho1, rho2)
        ll -= _integrate_rate(t_prev, t_ev, base_a, xi2, rho1, rho2)

        if kind == "H_GOAL":
            ll += np.log(_rate_at(t_ev, base_h, xi1, rho1, rho2))
            x += 1
        elif kind == "A_GOAL":
            ll += np.log(_rate_at(t_ev, base_a, xi2, rho1, rho2))
            y += 1
        t_prev = t_ev

    # 最後の得点から試合終了(t=1)までの区間
    sh, sa = _team_states(x, y)
    base_h = lam_k * th[sh]
    base_a = mu_k * th[sa]
    ll -= _integrate_rate(t_prev, 1.0, base_h, xi1, rho1, rho2)
    ll -= _integrate_rate(t_prev, 1.0, base_a, xi2, rho1, rho2)
    return ll


def negative_log_likelihood(theta, matches, idx):
    n_teams = len(idx)
    alpha, beta, gamma_h, rho1, rho2, th, xi1, xi2 = unpack_params(theta, n_teams)
    total = 0.0
    for m in matches:
        try:
            total += _match_loglik(m, idx, alpha, beta, gamma_h, rho1, rho2, th, xi1, xi2)
        except _InvalidRate:
            total += -1e9   # 強度が負になるときは大きなペナルティ
    if not np.isfinite(total):
        return 1e12
    return -total


def _count_invalid_matches(theta, matches, idx):
    """推定後のパラメータで、強度が0以下になる試合の数を数える"""
    n_teams = len(idx)
    alpha, beta, gamma_h, rho1, rho2, th, xi1, xi2 = unpack_params(theta, n_teams)
    n_invalid = 0
    for m in matches:
        try:
            _match_loglik(m, idx, alpha, beta, gamma_h, rho1, rho2, th, xi1, xi2)
        except _InvalidRate:
            n_invalid += 1
    return n_invalid

## 初期値

元のノートブックと同じ。攻撃力・守備力はチームの平均得点・平均失点から作り、倍率は1から始める。

In [19]:
def initial_theta(matches, idx):
    n_teams = len(idx)
    goals_for = np.zeros(n_teams)
    goals_against = np.zeros(n_teams)
    games = np.zeros(n_teams)
    total_goals = 0
    total_games = 0
    home_goals_total = 0
    away_goals_total = 0
    for m in matches:
        hi, aj = idx[m["home"]], idx[m["away"]]
        xg = sum(1 for t, k in m["events"] if k == "H_GOAL")
        yg = sum(1 for t, k in m["events"] if k == "A_GOAL")
        goals_for[hi] += xg
        goals_against[aj] += xg
        goals_for[aj] += yg
        goals_against[hi] += yg
        games[hi] += 1
        games[aj] += 1
        total_goals += xg + yg
        total_games += 1
        home_goals_total += xg
        away_goals_total += yg

    avg = total_goals / max(2 * total_games, 1)
    with np.errstate(divide="ignore", invalid="ignore"):
        att0 = np.where(games > 0, (goals_for / np.maximum(games, 1)) / avg, 1.0)
        def0 = np.where(games > 0, (goals_against / np.maximum(games, 1)) / avg, 1.0)
    att0 = np.clip(att0, 0.3, 3.0)
    def0 = np.clip(def0, 0.3, 3.0)

    a0 = np.log(att0)
    b0 = np.log(def0)

    gamma0 = home_goals_total / max(away_goals_total, 1)
    g0 = np.zeros(N_GLOBAL)
    g0[0] = np.log(max(gamma0, 0.1))   # gamma_h 以外は 0（倍率1）から始める
    return np.concatenate([a0, b0, g0])

## 推定

元のノートブックと同じく、L-BFGS-B → Powell の2段階で最適化する。
パラメータは20チームなら 20 + 20 + 9 = 49個（元のモデルより4個少ない）。

Powell が途中で得点強度が負になるところに入って、1段目より悪い値で終わることがある。
なので `recent_scorer_model.ipynb` と同じように、最後は「初期値・1段目・2段目」のうち一番尤度が高いものを使う。

In [20]:
def fit_dixon_robinson_shared(df, maxiter=1000, disp=False):
    """倍率を共通にしたモデルを推定する。

    戻り値:
        summary       : 対数尤度・AIC・BICなど
        team_params   : チームごとの攻撃力・守備力
        global_params : チーム以外のパラメータ
        res           : 最適化の結果
    """
    matches = prepare_matches(df)
    teams, idx = build_index(matches)
    n_teams = len(teams)

    theta0 = initial_theta(matches, idx)

    # パラメータが動ける範囲（xi 以外は log の値）
    bounds = [(-3, 3)] * n_teams + [(-3, 3)] * n_teams
    bounds += [(-3, 3)]              # gamma_h
    bounds += [(-3, 3), (-3, 3)]     # rho1, rho2
    bounds += [(-3, 3)] * N_STATES   # lead1, behind1, leadBig, behindBig
    bounds += [(-5, 5), (-5, 5)]     # xi1, xi2（log ではない）

    res1 = minimize(
        negative_log_likelihood, theta0, args=(matches, idx),
        method="L-BFGS-B", bounds=bounds,
        options={"maxiter": maxiter, "maxfun": maxiter * 50},
    )
    f0 = negative_log_likelihood(theta0, matches, idx)
    start2 = res1.x if res1.fun <= f0 else theta0
    res = minimize(
        negative_log_likelihood, start2, args=(matches, idx),
        method="Powell", bounds=bounds,
        options={"maxiter": maxiter * 20, "maxfev": maxiter * 200, "xtol": 1e-10, "ftol": 1e-12},
    )
    if disp:
        print(f"[stage1: L-BFGS-B] loglik={-res1.fun:.4f} success={res1.success}")
        print(f"[stage2: Powell]   loglik={-res.fun:.4f} success={res.success}")

    # Powell が強度が負になるところに入って、1段目より悪くなることがある
    # 初期値・1段目・2段目のうち一番尤度が高いものを使う
    candidates = [(f0, theta0, "initial"), (res1.fun, res1.x, "L-BFGS-B"), (res.fun, res.x, "Powell")]
    best_fun, best_x, best_stage = min(candidates, key=lambda c: c[0])
    if best_stage != "Powell":
        res.x, res.fun = best_x, best_fun
        if disp:
            print(f"[selected: {best_stage}] loglik={-best_fun:.4f}")

    theta = res.x
    alpha, beta, gamma_h, rho1, rho2, th, xi1, xi2 = unpack_params(theta, n_teams)

    n_params = len(theta)
    loglik = -res.fun
    aic = 2 * n_params - 2 * loglik
    # BIC の n は試合数にする
    bic = n_params * np.log(len(matches)) - 2 * loglik

    team_params = pd.DataFrame({"team": teams, "alpha_attack": alpha, "beta_defence": beta})
    global_params = pd.DataFrame({
        "parameter": PARAM_NAMES_GLOBAL,
        "estimate": [gamma_h, rho1, rho2] + [th[s] for s in STATES] + [xi1, xi2],
    })

    n_invalid = _count_invalid_matches(theta, matches, idx)
    message = str(res.message)
    if n_invalid > 0:
        message = (
        f"[WARNING] Even after convergence, {n_invalid} match(es) still have "
        f"non-positive scoring intensity, so a -1e9 penalty has leaked into "
        f"the log-likelihood. The log-likelihood/AIC/BIC values are not "
        f"reliable. " + message
    )

    summary = {
        "n_matches": len(matches), "n_teams": n_teams, "n_params": n_params,
        "log_likelihood": loglik, "AIC": aic, "BIC": bic,
        "converged": bool(res.success), "message": message,
    }

    print("==== モデル適合結果 ====")
    print(f"試合数: {summary['n_matches']}, チーム数: {summary['n_teams']}, パラメータ数: {summary['n_params']}")
    print(f"対数尤度: {summary['log_likelihood']:.3f}")
    print(f"AIC: {summary['AIC']:.3f}")
    print(f"BIC: {summary['BIC']:.3f}")
    print(f"収束: {summary['converged']} ({summary['message']})")
    print()
    print("---- チーム別パラメータ ----")
    print(team_params.to_string(index=False))
    print()
    print("---- 共通パラメータ ----")
    print(global_params.to_string(index=False))

    return summary, team_params, global_params, res

## 1ファイルで試す

Premier League 2015/16 で推定してみる。

In [21]:
df = pd.read_csv("../statsbomb_data/Premier_League/PL2015-2016_goals_and_red_cards.csv")
summary, team_params, global_params, res = fit_dixon_robinson_shared(df, disp=True)

[stage1: L-BFGS-B] loglik=-555.6425 success=True
[stage2: Powell]   loglik=-555.6425 success=True
==== モデル適合結果 ====
試合数: 380, チーム数: 20, パラメータ数: 49
対数尤度: -555.642
AIC: 1209.285
BIC: 1402.353
収束: True (Optimization terminated successfully.)

---- チーム別パラメータ ----
                team  alpha_attack  beta_defence
     AFC Bournemouth      0.856310      1.110399
             Arsenal      1.340643      0.515901
         Aston Villa      0.347877      1.242902
             Chelsea      1.218310      0.778309
      Crystal Palace      0.670857      0.763997
             Everton      1.232632      0.860054
      Leicester City      1.399452      0.452049
           Liverpool      1.297054      0.728377
     Manchester City      1.518986      0.654911
   Manchester United      0.934929      0.481316
    Newcastle United      0.788015      1.048480
        Norwich City      0.708147      1.060073
         Southampton      1.170551      0.556781
          Stoke City      0.784210      0.819292
     

## 全リーグ・全シーズンで推定する

全ファイルで推定して、AIC・BICの一覧とパラメータをcsvに保存する。
保存先は `dixon_robinson_shared_results`。

In [22]:
import glob
import os

DATA_DIR = "../statsbomb_data"
OUT_DIR = "./dixon_robinson_shared_results"
os.makedirs(OUT_DIR, exist_ok=True)

csv_paths = sorted(glob.glob(os.path.join(DATA_DIR, "**", "*_goals_and_red_cards.csv"), recursive=True))
print(f"{len(csv_paths)} 件のcsvが見つかりました")

results_all = []
for path in csv_paths:
    print("=" * 60)
    print(path)
    df_i = pd.read_csv(path)
    league = df_i["competition_name"].iloc[0] if "competition_name" in df_i.columns and len(df_i) else os.path.basename(path)
    season = df_i["season_name"].iloc[0] if "season_name" in df_i.columns and len(df_i) else ""
    tag = f"{league}_{season}".replace("/", "-").replace(" ", "_")
    try:
        summary_i, team_params_i, global_params_i, _ = fit_dixon_robinson_shared(df_i, disp=True)
        summary_i["league"] = league
        summary_i["season"] = season
        summary_i["file"] = os.path.basename(path)
        results_all.append(summary_i)
        team_params_i.to_csv(os.path.join(OUT_DIR, f"team_params_{tag}.csv"), index=False)
        global_params_i.to_csv(os.path.join(OUT_DIR, f"global_params_{tag}.csv"), index=False)
    except Exception as e:
        print("失敗:", e)

summary_all_df = pd.DataFrame(results_all)
summary_all_df.to_csv(os.path.join(OUT_DIR, "summary_all.csv"), index=False)
summary_all_df

11 件のcsvが見つかりました
../statsbomb_data/FA_Women's_Super_League/WSL2018-2019_goals_and_red_cards.csv
[stage1: L-BFGS-B] loglik=-100.4250 success=True
[stage2: Powell]   loglik=-100.3790 success=True
==== モデル適合結果 ====
試合数: 107, チーム数: 11, パラメータ数: 31
対数尤度: -100.379
AIC: 262.758
BIC: 345.616
収束: True (Optimization terminated successfully.)

---- チーム別パラメータ ----
                      team  alpha_attack  beta_defence
               Arsenal WFC      2.295087      0.722567
       Birmingham City WFC      0.955658      0.866109
Brighton & Hove Albion WFC      0.558651      1.796698
          Bristol City WFC      0.565840      1.658177
               Chelsea FCW      1.325121      0.673335
               Everton LFC      0.525437      1.649493
             Liverpool WFC      0.692959      1.804656
       Manchester City WFC      1.734695      0.897243
               Reading WFC      1.140571      1.543709
       West Ham United LFC      0.825329      1.748233
           Yeovil Town LFC      0.380652 

KeyboardInterrupt: 

## separated モデルとの尤度比検定

倍率を共通にしたことで、当てはまりが悪くなっていないかを確かめる。

このノートのモデル（shared）は、`dixon_robinson_model.ipynb` のモデル（separated）に次の制約を入れたものになっている。

| shared | separated で同じになるもの |
|---|---|
| lead1 | $\lambda_{home1}$ と $\mu_{away1}$ |
| behind1 | $\lambda_{away1}$ と $\mu_{home1}$ |
| leadBig | $\lambda_{homeBig}$ と $\mu_{awayBig}$ |
| behindBig | $\lambda_{awayBig}$ と $\mu_{homeBig}$ |

なので shared は separated の特別な場合（入れ子）で、尤度比検定が使える。

- 帰無仮説 $H_0$：倍率はホームとアウェイで共通（shared で十分）
- 対立仮説 $H_1$：倍率はホームとアウェイで違う（separated が必要）

$$LR = 2\,(\ell_{sep} - \ell_{shared})$$

$H_0$ のもとで $LR$ は自由度 $= 53 - 49 = 4$ のカイ二乗分布に近似的に従う。
p値が小さければ（5%未満なら）、倍率を共通にするのは無理があるということになる。

それぞれの推定はもう終わっているので、`summary_all.csv` 同士を比べるだけにする。

In [ ]:
from scipy.stats import chi2

sep_df = pd.read_csv("./dixon_robinson_pure_results/summary_all.csv")
shared_df = pd.read_csv("./dixon_robinson_shared_results/summary_all.csv")

# ファイル名でくっつける
lr_df = pd.merge(
    sep_df[["file", "league", "season", "n_matches", "n_params", "log_likelihood", "message"]],
    shared_df[["file", "n_params", "log_likelihood", "message"]],
    on="file", suffixes=("_sep", "_shared"),
)

lr_df["LR"] = 2 * (lr_df["log_likelihood_sep"] - lr_df["log_likelihood_shared"])
lr_df["df"] = lr_df["n_params_sep"] - lr_df["n_params_shared"]   # 4 になるはず
lr_df["p_value"] = chi2.sf(lr_df["LR"], lr_df["df"])

# 尤度にペナルティ(-1e9)が入っているものは比べられないので外す
bad = lr_df["message_sep"].str.contains("WARNING") | lr_df["message_shared"].str.contains("WARNING")
# separated の方が尤度が低いのはおかしい（最適化がうまくいっていない）ので、これも外す
bad = bad | (lr_df["LR"] < 0)
lr_df["use"] = ~bad
lr_df.loc[bad, ["LR", "p_value"]] = np.nan

print("検定に使えなかったもの:")
print(lr_df.loc[bad, ["league", "season"]].to_string(index=False))
print()

lr_df[["league", "season", "n_matches", "log_likelihood_sep", "log_likelihood_shared", "LR", "df", "p_value"]]

NameError: name 'pd' is not defined

### 全部まとめて検定する

リーグ・シーズンごとのデータは別々なので、LR と自由度をそのまま足して1つの検定にもできる。
1つずつだと試合数が少なくて差が出にくいので、まとめた方も見ておく。

In [ ]:
use_df = lr_df[lr_df["use"]]

LR_total = use_df["LR"].sum()
df_total = use_df["df"].sum()
p_total = chi2.sf(LR_total, df_total)

print(f"使ったデータ数: {len(use_df)}")
print(f"LR 合計: {LR_total:.3f}, 自由度: {df_total}, p値: {p_total:.4f}")
print(f"5%で有意になったもの: {(use_df['p_value'] < 0.05).sum()} / {len(use_df)}")

lr_df.to_csv(os.path.join(OUT_DIR, "lr_test_vs_separated.csv"), index=False)

NameError: name 'lr_df' is not defined

## separated モデルのパラメータを並べて比べる

`dixon_robinson_model.ipynb`（separated）で推定した倍率を、リーグ・シーズンごとに並べて見る。

上の表と同じように、チームから見て同じ状況になるものを隣に並べる。

| 状況 | ホームの倍率 | アウェイの倍率 |
|---|---|---|
| 1-0でリード | $\lambda_{10}$（lambda_home1） | $\mu_{01}$（mu_away1） |
| 0-1で負けている | $\lambda_{01}$（lambda_away1） | $\mu_{10}$（mu_home1） |
| 1-0以外でリード | lambda_homeBig | mu_awayBig |
| 0-1以外で負けている | lambda_awayBig | mu_homeBig |

ホームとアウェイの値が近ければ、倍率を共通にしても問題なさそうということになる。
パラメータは `dixon_robinson_pure_results/global_params_*.csv` に保存してあるものを読むだけにする。

In [13]:
import pandas as pd

SEP_DIR = "./dixon_robinson_pure_results"
sep_df = pd.read_csv(os.path.join(SEP_DIR, "summary_all.csv"))

# 同じ状況になる (ホームの倍率, アウェイの倍率) の組
pairs = [
    ("lambda_home1", "mu_away1"),      # 1-0でリード
    ("lambda_away1", "mu_home1"),      # 0-1で負けている
    ("lambda_homeBig", "mu_awayBig"),  # 1-0以外でリード
    ("lambda_awayBig", "mu_homeBig"),  # 0-1以外で負けている
]

rows = []
for i in range(len(sep_df)):
    league = sep_df["league"][i]
    season = sep_df["season"][i]
    # 保存したときと同じファイル名を作る
    tag = f"{league}_{season}".replace("/", "-").replace(" ", "_")
    g = pd.read_csv(os.path.join(SEP_DIR, f"global_params_{tag}.csv"))
    est = dict(zip(g["parameter"], g["estimate"]))

    row = {"league": league, "season": season}
    for home_name, away_name in pairs:
        row[home_name] = est[home_name]
        row[away_name] = est[away_name]
    # 推定がうまくいっていないもの（WARNING が出ているもの）に印をつける
    row["warning"] = "WARNING" in str(sep_df["message"][i])
    rows.append(row)

compare_df = pd.DataFrame(rows)
compare_df.round(3)

,league,season,lambda_home1,mu_away1,lambda_away1,mu_home1,lambda_homeBig,mu_awayBig,lambda_awayBig,mu_homeBig,warning
0,FA Women's Super League,2018/2019,0.872,0.837,0.880,1.152,0.841,0.982,1.095,0.753,False
1,FA Women's Super League,2019/2020,0.388,0.630,0.505,0.802,0.516,0.701,1.161,1.398,False
2,FA Women's Super League,2020/2021,1.113,0.844,1.298,0.950,1.266,1.026,0.923,1.187,False
3,FA Women's Super League,2023/2024,0.585,0.512,0.605,1.447,0.669,0.796,0.615,1.241,False
4,Frauen Bundesliga,2023/2024,1.026,0.690,0.834,0.701,0.869,0.924,0.999,1.034,True
5,Indian Super league,2021/2022,0.937,1.059,1.535,1.485,1.061,0.964,0.631,1.831,False
6,La Liga,2015/2016,0.916,0.799,1.169,0.775,0.849,0.822,0.764,0.979,False
7,Liga F,2023/2024,0.825,0.616,1.062,0.975,0.906,0.794,1.111,1.066,False
8,Ligue 1,2015/2016,1.021,1.035,1.085,1.173,1.020,1.046,1.185,0.906,False
9,Premier League,2015/2016,1.096,1.144,1.197,1.155,0.934,0.916,1.350,1.291,False


## サッカーの試合のスコア予測モデルに関する研究は1982年にMaherによって基盤となる確率モデルが提案されて以来、得点に適用するためにさまざまな改良がなされてきた。その中でも、1998年にDixon&Robinsonによって提案された90分間の得点の遷移に二変量の出生過程を仮定したモデルは、試合状況に応じてハザードを変化させることができるという点で、サッカーの得点に適したモデルであると言える。Dixon&Robinsonモデルは、Maherのモデルを基盤とし、各得点状況に応じたハザードの変化と一次関数的なハザードの向上を仮定している。果たしてこのような分類は得点の遷移に寄与する要因を正しく反映できているのだろうかと考える。本研究は得点間の待ち時間に対し、さまざまな経験的な分析をし、より得点遷移の特徴を捉えた出生過程モデルの構築を目指している。

## サッカーの試合のスコア予測モデルに関する研究は1982年にMaherによって基盤となる確率モデルが提案されて以来、得点に適用するためにさまざまな改良がなされてきた。その中でも、1998年にDixon&Robinsonによって提案された90分間の得点の遷移に二変量の出生過程を仮定したモデルは、試合状況に応じてハザードを変化させることができるという点で、サッカーの得点に適したモデルであると言える。Dixon&Robinsonモデルは、Maherのモデルを基盤とし、各得点状況を考慮した上で時間軸に線形的に増加するハザード関数を仮定している。各得点状況及びの変化を仮定している。しかしながら、経験的な分析によると、必ずしもこの仮定では捉えきれない。本研究は得点間の待ち時間に対し、さまざまな経験的な分析をし、より得点遷移の特徴を捉えた出生過程モデルの構築を目指している。

## Submission Guidelines :
## Due Date : October 11, 2026 (Sunday)
## Poster Abstract Limit: Maximum of 4 pages.
## Poster Dimensions: Standard A1 size (594mm*841mm) is recommended.

## via the GitHub repository statsbomb/open-data)

## I've attached a logo.png to this email that can be used - if there are any issues as long as the credit exists I think that's the main thing